## Retrieve Under Utilized Lambda Functions

In [7]:
import boto3
from datetime import datetime, timedelta
import pytz
import os
import pandas as pd

from botocore.exceptions import NoCredentialsError

# Get credentials from environment variables
aws_access_key_id = os.getenv('AWS_ACCESS_KEY_ID')
aws_secret_access_key = os.getenv('AWS_SECRET_ACCESS_KEY')
aws_session_token = os.getenv('AWS_SESSION_TOKEN')  # Optional, for temporary credentials
region_name = os.getenv('AWS_DEFAULT_REGION', 'ap-southeast-1')  # Default to 'ap-southeast-1' if not set

In [9]:
# Validate credentials
if not aws_access_key_id or not aws_secret_access_key:
    raise NoCredentialsError

# Initialize clients
lambda_client = boto3.client(
    'lambda',
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token,
    region_name=region_name
)


cloudwatch_client = boto3.client(
    'cloudwatch',
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token,
    region_name=region_name
)

# set Singapore timezone
timezone = pytz.timezone('Asia/Singapore')


# Retrieve lambda functions with their invokation metrics

ChatGPT Prompt: from the lambda metrics, retrieve the metrics, i.e. last time invoked (in date granularity), total invokation in the last month, and sort them


In [10]:
# Get the current datetime
now = datetime.now(tz=timezone)
start_time = now - timedelta(days=90)  # Start time: 30 days ago

def get_lambda_metrics():
    functions = lambda_client.list_functions()['Functions']
    results = []

    for function in functions:
        function_name = function['FunctionName']

        # Get the total invocations in the last 30 days
        invocations = cloudwatch_client.get_metric_statistics(
            Namespace='AWS/Lambda',
            MetricName='Invocations',
            Dimensions=[
                {'Name': 'FunctionName', 'Value': function_name}
            ],
            StartTime=start_time,
            EndTime=now,
            Period=14*86400,  # 14 days: 2 weeks
            Statistics=['Sum']
        )

        # Calculate total invocations
        total_invocations = sum(dp['Sum'] for dp in invocations['Datapoints'])

        # Get the last invocation time
        if invocations['Datapoints']:
            last_invocation_time = max(dp['Timestamp'] for dp in invocations['Datapoints'])
        else:
            last_invocation_time = None

        results.append({
            'FunctionName': function_name,
            'LastInvocationTime': last_invocation_time,
            'TotalInvocations': total_invocations
        })

    # Sort by total invocations descending, then by last invocation time descending
    results.sort(
        key=lambda x: (
            -x['TotalInvocations'], x['LastInvocationTime'] if x['LastInvocationTime'] else datetime.min
        ),
        reverse=True
    )
    return results

In [11]:
metrics = get_lambda_metrics()

# Convert results to a DataFrame
df = pd.DataFrame(metrics)
df.head()

# Save DataFrame to CSV locally
csv_file_path = "lambda_metrics.csv"
df.to_csv(csv_file_path, index=False)


print(f"Metrics saved to {csv_file_path}.")


Metrics saved to lambda_metrics.csv.


### Combine with SteamPipe Analysis

In [26]:
csv_file_path = "dev_tags_compliance_dashboard_resource_details_aws_lambda_function_table_20250103T111938.csv"
df_steampipe = pd.read_csv(csv_file_path)
# df_steampipe.drop(columns=["account_id", "region"], inplace=True)
df_steampipe = df_steampipe[["identifier"]]
df_steampipe["NonCompliant"] = True
df_steampipe.head(1)

,identifier,NonCompliant
0,InvokeImageModerationLoadTest,True


In [57]:
df_merged = df_steampipe.merge(df, left_on="identifier", right_on="FunctionName", how="left")
df_merged["FunctionName"] = df_merged["FunctionName"].fillna("")
df_merged["CloudWatch"] = df_merged["FunctionName"].apply(
    lambda x: "Available" if x else "Not Available"
)
df_merged.drop(columns=["FunctionName"], inplace=True)

In [60]:
df_merged.head(1)

,identifier,NonCompliant,LastInvocationTime,TotalInvocations,CloudWatch
0,InvokeImageModerationLoadTest,True,NaT,NaN,Not Available
